##Load Library dan Dataset

In [1]:
!pip install datasets soundfile librosa -q

In [2]:
import numpy as np
import librosa
import soundfile as sf
from io import BytesIO
from datasets import load_dataset, concatenate_datasets, Dataset, Audio

In [3]:
def load_n_samples(name, label, n=400):
    stream = load_dataset("cgeorgiaw/animal-sounds", name, streaming=True)["train"]
    samples = []
    for i, example in enumerate(stream):
        if i >= n:
            break
        samples.append({
            "audio": example["audio"],
            "label": label
        })
    return Dataset.from_list(samples)

birds       = load_n_samples("birds",       "birds")
macaque     = load_n_samples("macaque",     "macaque")
giant_otter = load_n_samples("giant_otter", "giant_otter")
orca        = load_n_samples("orca",        "orca")
zebra_finch = load_n_samples("zebra_finch", "zebra_finch")

# Gabungkan
dataset = concatenate_datasets([birds, macaque, giant_otter, orca, zebra_finch])
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/595 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3406 [00:00<?, ?it/s]

Total sampel: 2000
Kolom      : ['audio', 'label']


In [4]:
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

Total sampel: 2000
Kolom      : ['audio', 'label']


##Preprocessing


###1. Resampling

In [5]:
# Cast audio supaya bisa di-decode
dataset = dataset.cast_column("audio", Audio())

# Cek sample rate tiap kelas
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

birds                → sample rate: 44100 Hz
macaque              → sample rate: 24414 Hz
giant_otter          → sample rate: 96000 Hz
orca                 → sample rate: 44100 Hz
zebra_finch          → sample rate: 44100 Hz


In [6]:
# Resample semua ke 22050 Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

# Verifikasi
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

birds                → sample rate: 22050 Hz
macaque              → sample rate: 22050 Hz
giant_otter          → sample rate: 22050 Hz
orca                 → sample rate: 22050 Hz
zebra_finch          → sample rate: 22050 Hz
